In [1]:
from parsing_library import *

# Repeated experiments

In [5]:
## ---- from saved calculate_coverage.py json files -----

import os
import json

saved_coverage_dir = "../../coverage_results_ten_V100K5_zipf"

# Dictionary to store sums and counts for averaging
size_sums = {}
#size_counts = {}
count = 0
# Traverse through subdirectories and collect JSON files
for root, dirs, files in os.walk(saved_coverage_dir):
    for file in files:
        if file.endswith(".json"):
            json_path = os.path.join(root, file)
            print(json_path)
            with open(json_path, 'r') as f:
                data = json.load(f)

            # Sum values for each size
            for size, value in data.items():
                size = int(size) 
                if size not in size_sums:
                    size_sums[size] = 0
                    if size_sums[size] == 0 and count > 0:
                        print(f'missing size? {json_path}, {size}')
                size_sums[size] += value
            count += 1

averages = [f"{(size_sums[size] / count):.1f}" for size in sorted(size_sums.keys())]
result_string = " & ".join(averages)

print(result_string)


../../coverage_results_ten_V100K5_zipf/2-zipf/coverage_2.json
../../coverage_results_ten_V100K5_zipf/7-zipf/coverage_7.json
../../coverage_results_ten_V100K5_zipf/8-zipf/coverage_8.json
../../coverage_results_ten_V100K5_zipf/6-zipf/coverage_6.json
../../coverage_results_ten_V100K5_zipf/4-zipf/coverage_4.json
../../coverage_results_ten_V100K5_zipf/3-zipf/coverage_3.json
../../coverage_results_ten_V100K5_zipf/9-zipf/coverage_9.json
../../coverage_results_ten_V100K5_zipf/10-zipf/coverage_10.json
../../coverage_results_ten_V100K5_zipf/5-zipf/coverage_5.json
../../coverage_results_ten_V100K5_zipf/1-zipf/coverage_1.json
74.4 & 72.6 & 69.0 & 68.2 & 64.5 & 58.6 & 51.9


In [ ]:
from parsing_library import *

# Assume:
# - ten_2d_datasets contains datasets, ex: ten_2d_datasets/1.json
# - ten_2d_datasets/inference_method/1 are the fit folders

main_folder = '../../data/ten_V100K5-zipf'#'ten_2d_datasets' 
#saved_fit_folder = 'ten_V100K5/nofix/vi/'
saved_fit_folder = '../../results_ten_V100K5_zipf/nofix/vi'

data_folder = '../../data/ten_V100K5-zipf'
inference_method = 'vi'

fit_folder_base_name = "%i-zipf"
# fit_folder_base_name = %i"
base_data_name = "%i-zipf.json"
#base_data_name = "%i.json"


experiments_ids = [1,2,3, 4, 5, ,6, 7, 8, 9, 10]
#sizes = [100, 200, 500, 1000, 5000, 20000, 50000, 100000]
sizes = [1000, 2000, 5000, 10000, 20000, 50000, 100000]

coverage_per_size = {size: [] for size in sizes}

for size in sizes:
    print(f"Processing size {size}...")
    
    for repetition in experiments_ids:
        covered_count = 0
        
        # data
        data_path = os.path.join(data_folder,base_data_name%repetition)
        with open(data_path) as f:
            data = json.load(f)

        true_theta = np.array(data['theta'])
        vocabulary = data['vocabulary']
        word_pairs = list(itertools.combinations(vocabulary, 2))
        total_pairs = len(word_pairs) * 2

        # inference fit path
        saved_fit_dir_i = os.path.join(saved_fit_folder, fit_folder_base_name%repetition)
        
        if not inference_method == 'gibbs':
            fit = load_fit_by_size(size, saved_fit_dir_i) #load_fit

        if inference_method == 'hmc':
            estimated_word_vectors = fit['word_vectors']
            estimated_context_vectors = fit['context_vectors']
            num_samples = estimated_word_vectors.shape[2]
        elif inference_method == 'vi':
            estimated_word_vectors, estimated_context_vectors = extract_word_and_context_vectors_vi(fit.variational_sample_pd, vocabulary)
            num_samples = estimated_word_vectors.shape[2]
        elif inference_method in ('laplace', 'map'):
            estimated_word_vectors, estimated_context_vectors = extract_word_and_context_vectors_vi(fit.draws_pd(), vocabulary)
            num_samples = estimated_word_vectors.shape[2]
        elif inference_method == 'gibbs':
            filepath = os.path.join(saved_fit_dir_i, f'gibbs-samples-N-{size}.csv.fix')
            estimated_word_vectors, estimated_context_vectors = extract_word_and_context_vectors_gibbs(filepath, vocabulary)
            num_samples = estimated_word_vectors.shape[2]

        theta_samples = []
        for i in range(num_samples):
            word_vectors = estimated_word_vectors[:, :, i]
            context_vectors = estimated_context_vectors[:, :, i]
            theta = np.vstack((word_vectors, context_vectors))
            theta_samples.append(theta)

        # loop through word pairs and compute coverage
        for word1, word2 in word_pairs:
            vi = vocabulary[word1]
            wi = vocabulary[word2]
            true_p = calculate_p(vi, wi, true_theta, len(vocabulary))

            eta_samples = [calculate_p(vi, wi, theta, len(vocabulary)) for theta in theta_samples]
            lower_ci, upper_ci = credible_interval(eta_samples)

            if lower_ci <= true_p <= upper_ci:
                covered_count += 1

            true_p = calculate_p(wi, vi, true_theta, len(vocabulary))

            eta_samples = [calculate_p(wi, vi, theta, len(vocabulary)) for theta in theta_samples]
            lower_ci, upper_ci = credible_interval(eta_samples)

            if lower_ci <= true_p <= upper_ci:
                covered_count += 1

        coverage_percentage = (covered_count / total_pairs) * 100
        coverage_per_size[size].append(coverage_percentage)

# Average the coverage for each size
average_coverage_per_size = {size: np.mean(coverage_per_size[size]) for size in sizes}


coverage_stats_per_size = {}
for size in sizes:
    coverage_list = coverage_per_size[size]
    mean_coverage = np.mean(coverage_list)
    std_coverage = np.std(coverage_list)  # Standard deviation
    sem_coverage = std_coverage / np.sqrt(len(coverage_list))  # SEM
    coverage_stats_per_size[size] = {'mean': mean_coverage, 'std': std_coverage, 'sem': sem_coverage}


for size, stats in coverage_stats_per_size.items():
    print(f"Data size: {size}, Average coverage: {stats['mean']:.2f}%, ± {stats['std']:.2f}% (std), ± {stats['sem']:.2f}% (SEM)")

